# Coordinator — Stage 1 pipeline playground

Runs the full 4-node ingestion pipeline end-to-end via a single `graph.ainvoke` call.

```
[FILE]  →  Node 1 (reception)  →  Node 2 (format)  →  extract  →  Node 3 (content)  →  Node 4 (duplicate)  →  ACCEPTED / REJECTED / REVIEW
```

Each node writes its result into a shared `JobState` dict; conditional edges route
to `accept`, `reject`, or `review` based on what each node returns.

> **Kernel**: select `.venv` (Python 3.10.11) in the top-right kernel picker.
>
> **Node 3 note**: content validation requires an SLM (Phi-4-mini GGUF) and a
> language detector. Cells 4–11 patch both with lightweight mocks so the notebook
> runs without downloading any model. Section 12 shows real-file behaviour.

## 1 — Imports

In [1]:
import asyncio
from pathlib import Path

from sqlalchemy.ext.asyncio import AsyncSession, async_sessionmaker, create_async_engine

import classiflow.ingesta.nodes.node3_content_validation as _n3_mod
from classiflow.database.base import Base
from classiflow.database.repositories.audit import SqlAuditRepository
from classiflow.database.repositories.hash import SqlHashRepository
from classiflow.events.broadcaster import EventBroadcaster
from classiflow.ingesta.coordinator import build_coordinator
from classiflow.ingesta.llm_provider import MockLlm
from classiflow.ingesta.mime import detect_mime
from classiflow.ingesta.nodes import ContentValidationNode, FileReceptionNode
from classiflow.ingesta.nodes.node2_format_validation import FormatValidationNode
from classiflow.ingesta.nodes.node4_duplicate_control import DuplicateControlNode, EmbeddingStore
from classiflow.services.audit.service import AuditService

print("imports OK")

c:\Users\leona\source\repos\Trabajo-Integrador\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imports OK


## 2 — Database setup

In [2]:
import classiflow.settings as _settings_mod

_project_root = Path(_settings_mod.__file__).parents[2]
_db_path = _project_root / "data" / "classiflow.db"
DB_URL = f"sqlite+aiosqlite:///{_db_path.as_posix()}"

# Always start with a fresh database so hash/duplicate state from prior sessions is gone
if _db_path.exists():
    _db_path.unlink()

engine = create_async_engine(DB_URL, echo=False)
session_factory = async_sessionmaker(engine, expire_on_commit=False)

async with engine.begin() as conn:
    await conn.run_sync(Base.metadata.create_all)

print(f"DB path : {_db_path}")
print("database ready (fresh)")

DB path : C:\Users\leona\source\repos\Trabajo-Integrador\data\classiflow.db
database ready (fresh)


## 3 — Node 3 mock helpers

Node 3 calls two external components at runtime:

| Component | Real behaviour | Mock behaviour |
|-----------|---------------|----------------|
| `_get_detector()` | loads `lingua-language-detector` model | returns a stub that always says `"es"` |
| `get_llm_langchain(path)` | loads a GGUF model from disk | returns `MockLlm` with a preset JSON response |

Both are module-level callables — replacing them here patches Node 3 for the
entire notebook session without touching any production code.

In [3]:
from dataclasses import dataclass

_SLM_ACCEPT = '{"is_legitimate": true,  "confidence": 0.92, "reasoning": "official doc"}'
_SLM_REJECT = '{"is_legitimate": false, "confidence": 0.88, "reasoning": "not official"}'


@dataclass
class _MockIsoCode:
    name: str


@dataclass
class _MockLanguage:
    iso_code_639_1: _MockIsoCode


class _MockDetector:
    def __init__(self, iso: str) -> None:
        self._iso = iso

    def detect_language_of(self, _text: str) -> _MockLanguage:
        return _MockLanguage(_MockIsoCode(self._iso))


_n3_mod.get_llm_langchain = lambda _path: MockLlm(response=_SLM_ACCEPT)  # type: ignore[attr-defined]

print("Node 3 mocks active -- language injected via constructor, SLM=accept")

Node 3 mocks active -- language injected via constructor, SLM=accept


## 4 — Pipeline helper

`run_pipeline` builds and runs the coordinator for a single document.
A shared `EmbeddingStore` accumulates the FAISS index across all calls
so that duplicate detection works across the sections below.

In [4]:
# Minimal valid PDF bytes — passes Node 1 and Node 2 without a real document
_MINIMAL_PDF = (
    b"%PDF-1.4\n1 0 obj\n<< /Type /Catalog >>\nendobj\n"
    b"xref\n0 1\n0000000000 65535 f\ntrailer\n<< /Size 1 >>\nstartxref\n9\n%%EOF"
)

# Representative Spanish municipal text used as the extracted content
SPANISH_TEXT = (
    "El Concejo Municipal de Rosario sanciona la siguiente ordenanza: "
    "Artículo 1º — Apruébase el presupuesto municipal para el ejercicio fiscal "
    "correspondiente al año en curso, conforme al detalle que se adjunta como Anexo I "
    "de la presente norma. Artículo 2º — El Departamento Ejecutivo Municipal adoptará "
    "las medidas necesarias para su implementación y seguimiento."
)

# Shared store — keeps the FAISS index alive across all cells below
_shared_store = EmbeddingStore()


def _make_nodes(
    session: AsyncSession, broadcaster: EventBroadcaster
) -> tuple[FileReceptionNode, FormatValidationNode, ContentValidationNode, DuplicateControlNode]:
    audit = AuditService(SqlAuditRepository(session))
    return (
        FileReceptionNode(audit=audit, broadcaster=broadcaster, mime_detector=detect_mime),
        FormatValidationNode(audit=audit, broadcaster=broadcaster),
        ContentValidationNode(
            audit=audit,
            broadcaster=broadcaster,
            language_detector=_MockDetector("es"),
        ),
        DuplicateControlNode(
            hash_repo=SqlHashRepository(session),
            audit=audit,
            broadcaster=broadcaster,
            embedding_store=_shared_store,
        ),
    )


async def run_pipeline(
    job_id: str,
    filename: str,
    file_bytes: bytes | None,
    *,
    extracted_text: str = SPANISH_TEXT,
) -> dict:
    broadcaster = EventBroadcaster()
    async with session_factory() as session:
        graph = build_coordinator(
            *_make_nodes(session, broadcaster),
            text_extractor=lambda _: extracted_text,
        )
        result = await graph.ainvoke({
            "job_id": job_id,
            "filename": filename,
            "file_bytes": file_bytes,
        })
        await session.commit()
    return result


def _print_result(result: dict) -> None:
    status = result.get("final_status", "—")
    icon = {"accepted": "✅", "rejected": "❌", "review": "⚠️"}.get(status, "?")
    print(f"  {icon} final_status : {status}")
    if result.get("rejection_reason"):
        print(f"  rejection    : {result['rejection_reason']}")
    for key in ("reception", "format_validation", "content_validation", "duplicate_control"):
        r = result.get(key)
        if r is not None:
            print(f"  {key:<24}: passed={r.passed}")


print("pipeline helper ready")

pipeline helper ready


## 5 — Happy path: ordinanza accepted

A valid PDF that passes all four nodes and reaches `final_status = accepted`.

- Node 1: file present, SHA-256 computed, MIME detected as `application/pdf`
- Node 2: PDF extension matches MIME → rule-based accept
- Node 3: text is long enough, language is Spanish, SLM says legitimate
- Node 4: first submission → no duplicate found

In [5]:
result = await run_pipeline(
    job_id="coord-demo-001",
    filename="ordenanza_presupuesto_2026.pdf",
    file_bytes=_MINIMAL_PDF,
)

print("=== Ordinanza — first submission ===")
_print_result(result)
print()
print(f"  sha256 (prefix) : {result['reception'].sha256[:24]}…")
print(f"  detected_mime   : {result['reception'].detected_mime}")
print(f"  char_count      : {result['content_validation'].char_count}")
print(f"  similarity_score: {result['duplicate_control'].similarity_score:.4f}")

2026-08-06 21:49:36.104 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-001 node=node1_file_reception event=passed
2026-08-06 21:49:36.104 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-001 node=node2_format_validation event=passed
2026-08-06 21:49:36.115 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-001 node=node3_content_validation event=passed
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4295.73it/s]
2026-08-06 21:49:40.247 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-001 node=node4_duplicate_control event=passed


=== Ordinanza — first submission ===
  ✅ final_status : accepted
  reception               : passed=True
  format_validation       : passed=True
  content_validation      : passed=True
  duplicate_control       : passed=True

  sha256 (prefix) : 30fdc7230755e392c94368b7…
  detected_mime   : application/pdf
  char_count      : 361
  similarity_score: 0.0000


## 6 — Exact duplicate rejected

Submitting the same file a second time. Node 4 finds the SHA-256 in its store
and rejects immediately — the embedding index is never consulted.

Expected: `final_status = rejected`, `duplicate_type = exact`, `similarity_score = 1.0`.

In [6]:
result = await run_pipeline(
    job_id="coord-demo-002",
    filename="ordenanza_presupuesto_2026_copia.pdf",  # different name, same bytes
    file_bytes=_MINIMAL_PDF,
)

print("=== Same file, different name ===")
_print_result(result)
print()
print(f"  is_duplicate    : {result['duplicate_control'].is_duplicate}")
print(f"  duplicate_type  : {result['duplicate_control'].duplicate_type}")
print(f"  similarity_score: {result['duplicate_control'].similarity_score:.4f}")

2026-08-06 21:49:40.292 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-002 node=node1_file_reception event=passed
2026-08-06 21:49:40.292 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-002 node=node2_format_validation event=passed
2026-08-06 21:49:40.303 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-002 node=node3_content_validation event=passed
2026-08-06 21:49:40.306 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-002 node=node4_duplicate_control event=failed


=== Same file, different name ===
  ❌ final_status : rejected
  rejection    : Exact duplicate: SHA-256 already in store
  reception               : passed=True
  format_validation       : passed=True
  content_validation      : passed=True
  duplicate_control       : passed=False

  is_duplicate    : True
  duplicate_type  : exact
  similarity_score: 1.0000


## 7 — Near-duplicate (semantic) rejected

A paraphrase of the same ordinanza has a different SHA-256 (passes the hash check)
but high cosine similarity in the embedding space.

Expected: `final_status = rejected`, `duplicate_type = semantic`, `similarity_score ≥ 0.85`.

In [7]:
# Different bytes → different SHA-256; but semantically similar text
_MINIMAL_PDF_V2 = _MINIMAL_PDF + b"\n%% version 2"

SPANISH_TEXT_PARAPHRASE = (
    "El Concejo Municipal de la ciudad de Rosario aprueba la siguiente ordenanza: "
    "Artículo 1° — Se aprueba el presupuesto municipal para el año fiscal en curso, "
    "de acuerdo con el detalle incorporado como Anexo I a la presente norma. "
    "Artículo 2° — El Ejecutivo Municipal dispondrá las acciones necesarias para su aplicación."
)

result = await run_pipeline(
    job_id="coord-demo-003",
    filename="ordenanza_presupuesto_2026_v2.pdf",
    file_bytes=_MINIMAL_PDF_V2,
    extracted_text=SPANISH_TEXT_PARAPHRASE,
)

print("=== Paraphrase of same ordinanza ===")
_print_result(result)
print()
print(f"  is_duplicate    : {result['duplicate_control'].is_duplicate}")
print(f"  duplicate_type  : {result['duplicate_control'].duplicate_type}")
print(f"  similarity_score: {result['duplicate_control'].similarity_score:.4f}")

2026-08-06 21:49:40.362 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-003 node=node1_file_reception event=passed
2026-08-06 21:49:40.366 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-003 node=node2_format_validation event=passed
2026-08-06 21:49:40.373 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-003 node=node3_content_validation event=passed
2026-08-06 21:49:40.407 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-003 node=node4_duplicate_control event=failed


=== Paraphrase of same ordinanza ===
  ❌ final_status : rejected
  rejection    : Near-duplicate: cosine similarity 0.870
  reception               : passed=True
  format_validation       : passed=True
  content_validation      : passed=True
  duplicate_control       : passed=False

  is_duplicate    : True
  duplicate_type  : semantic
  similarity_score: 0.8695


## 8 — Unrelated document accepted

A water-network technical report is on a completely different topic —
low cosine similarity, unique SHA-256. It passes all nodes.

Expected: `final_status = accepted`, `similarity_score < 0.85`.

In [8]:
_MINIMAL_PDF_V3 = _MINIMAL_PDF + b"\n%% informe agua"

INFORME_AGUA = (
    "Informe técnico sobre el estado de la red de agua potable del distrito norte. "
    "Se detectaron filtraciones en los tramos comprendidos entre las calles Córdoba "
    "y Corrientes. Se recomienda el reemplazo urgente de 200 metros de cañería "
    "en el sector afectado, con inicio de obras programado para el próximo trimestre."
)

result = await run_pipeline(
    job_id="coord-demo-004",
    filename="informe_red_agua_norte_2026.pdf",
    file_bytes=_MINIMAL_PDF_V3,
    extracted_text=INFORME_AGUA,
)

print("=== Unrelated document ===")
_print_result(result)
print()
print(f"  similarity_score: {result['duplicate_control'].similarity_score:.4f}  (should be < 0.85)")

2026-08-06 21:49:40.440 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-004 node=node1_file_reception event=passed
2026-08-06 21:49:40.444 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-004 node=node2_format_validation event=passed
2026-08-06 21:49:40.447 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-004 node=node3_content_validation event=passed
2026-08-06 21:49:40.488 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-004 node=node4_duplicate_control event=passed


=== Unrelated document ===
  ✅ final_status : accepted
  reception               : passed=True
  format_validation       : passed=True
  content_validation      : passed=True
  duplicate_control       : passed=True

  similarity_score: 0.0000  (should be < 0.85)


## 9 — Non-legitimate content goes to review

Switch the SLM mock to return `is_legitimate: false`.
Node 3 sends the document to the human review queue instead of rejecting outright.

Expected: `final_status = review`.

In [9]:
# Temporarily switch SLM to reject mode
_n3_mod.get_llm_langchain = lambda _path: MockLlm(response=_SLM_REJECT)  # type: ignore[attr-defined]

_MINIMAL_PDF_V4 = _MINIMAL_PDF + b"\n%% spam doc"

result = await run_pipeline(
    job_id="coord-demo-005",
    filename="documento_spam.pdf",
    file_bytes=_MINIMAL_PDF_V4,
)

print("=== Non-legitimate content ===")
_print_result(result)
print()
print(f"  needs_agent_review: {result['content_validation'].needs_agent_review}")

# Restore accept mode for the remaining sections
_n3_mod.get_llm_langchain = lambda _path: MockLlm(response=_SLM_ACCEPT)  # type: ignore[attr-defined]
print("\nSLM restored to accept mode")

2026-08-06 21:49:40.530 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-005 node=node1_file_reception event=passed
2026-08-06 21:49:40.530 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-005 node=node2_format_validation event=passed
2026-08-06 21:49:40.540 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-005 node=node3_content_validation event=failed


=== Non-legitimate content ===
  ⚠️ final_status : review
  rejection    : SLM: not official
  reception               : passed=True
  format_validation       : passed=True
  content_validation      : passed=False

  needs_agent_review: True

SLM restored to accept mode


## 10 — Rejection cases

Three hard failures that stop the pipeline at Node 1 — no subsequent nodes run.

| Case | Why rejected |
|------|--------------|
| Missing file | `file_bytes` is `None` |
| Empty file | `file_bytes` is `b""` |
| Wrong format | `.exe` extension rejected by Node 2 |

In [10]:
cases = [
    ("missing file", "doc.pdf", None),
    ("empty file", "doc.pdf", b""),
    ("wrong format", "mal.exe", _MINIMAL_PDF + b"\n%% exe"),
]

print("=== Rejection cases ===")
for label, filename, data in cases:
    result = await run_pipeline(
        job_id=f"coord-demo-rej-{label.replace(' ', '-')}",
        filename=filename,
        file_bytes=data,
        extracted_text="",
    )
    status = result.get("final_status")
    reason = result.get("rejection_reason") or "—"
    icon = "✅" if status == "accepted" else "❌"
    print(f"  {icon}  {label:<16}  status={status}  reason='{reason}'")

2026-08-06 21:49:40.589 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-rej-missing-file node=node1_file_reception event=failed


=== Rejection cases ===
  ❌  missing file      status=rejected  reason='No file provided'


2026-08-06 21:49:40.613 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-rej-empty-file node=node1_file_reception event=failed
2026-08-06 21:49:40.632 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-rej-wrong-format node=node1_file_reception event=passed


  ❌  empty file        status=rejected  reason='File is empty'


llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
2026-08-06 21:50:00.314 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-demo-rej-wrong-format node=node2_format_validation event=failed


  ❌  wrong format      status=rejected  reason='SLM: The file extension '.exe' suggests that the document is an executable, which can pose serious security threats if executed. This format has been identified as a high risk and thus cannot be accepted.'


## 11 — SSE events in real time

The coordinator emits a `STARTED` and `PASSED`/`FAILED` event for each node
as it executes. Subscribe to the broadcaster before calling `ainvoke` to
capture the full stream.

In [11]:
_MINIMAL_PDF_SSE = _MINIMAL_PDF + b"\n%% sse demo"

broadcaster_sse = EventBroadcaster()
events: list = []


async def _collect() -> None:
    async for event in broadcaster_sse.subscribe("coord-sse-001"):
        events.append(event)
        print(f"  SSE → node={event.node:<35}  status={event.status}")


async with session_factory() as session:
    graph_sse = build_coordinator(
        *_make_nodes(session, broadcaster_sse),
        text_extractor=lambda _: SPANISH_TEXT,
    )

    collect_task = asyncio.create_task(_collect())
    await asyncio.sleep(0)  # yield so _collect() subscribes before ainvoke emits

    sse_result = await graph_sse.ainvoke({
        "job_id": "coord-sse-001",
        "filename": "decreto_designacion_2026.pdf",
        "file_bytes": _MINIMAL_PDF_SSE,
    })
    await broadcaster_sse.close("coord-sse-001")
    await collect_task
    await session.commit()

print(f"\ncollected {len(events)} events  |  final_status={sse_result.get('final_status')}")

2026-08-06 21:50:00.368 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-sse-001 node=node1_file_reception event=passed
2026-08-06 21:50:00.373 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-sse-001 node=node2_format_validation event=passed
2026-08-06 21:50:00.380 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-sse-001 node=node3_content_validation event=passed
2026-08-06 21:50:00.477 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-sse-001 node=node4_duplicate_control event=failed


  SSE → node=node1_file_reception                 status=started
  SSE → node=node1_file_reception                 status=passed
  SSE → node=node2_format_validation              status=started
  SSE → node=node2_format_validation              status=passed
  SSE → node=node3_content_validation             status=started
  SSE → node=node3_content_validation             status=passed
  SSE → node=node4_duplicate_control              status=started
  SSE → node=node4_duplicate_control              status=failed

collected 8 events  |  final_status=rejected


## 12 — Real files from disk

Drop any PDF into `src/classiflow/playground/samples/` and run this cell.

> **Text extraction note**: MarkItDown is not yet integrated. The coordinator
> uses a UTF-8 fallback extractor that rarely produces usable text from binary
> PDFs. Real PDFs will usually be rejected at Node 3 (`requires_ocr = True`)
> until MarkItDown is added as a dependency. Text-based files (`.txt`, `.md`)
> pass Node 3 correctly with the current extractor.
>
> The Node 3 SLM mock is still active, so language and legitimacy checks use
> the stub — only text length and MIME/format checks reflect real file content.

In [12]:
import IPython.display as ipyd
from IPython.display import HTML

samples_dir = _project_root / "src" / "classiflow" / "playground" / "samples"
if not samples_dir.is_dir():
    samples_dir = None

print(samples_dir)
if samples_dir is None or not list(samples_dir.iterdir()):
    print("No files found. Drop a PDF into src/classiflow/playground/samples/ and re-run.")
else:
    for file_path in sorted(samples_dir.iterdir()):
        file_bytes = file_path.read_bytes()

        broadcaster_real = EventBroadcaster()
        async with session_factory() as session:
            # Use the real (UTF-8 fallback) extractor — no text injection
            graph_real = build_coordinator(*_make_nodes(session, broadcaster_real))
            real_result = await graph_real.ainvoke({
                "job_id": f"coord-real-{file_path.stem}",
                "filename": file_path.name,
                "file_bytes": file_bytes,
            })
            await session.commit()

        status = real_result.get("final_status", "—")
        reason = real_result.get("rejection_reason") or "—"
        _colors = {"accepted": "#2e7d32", "rejected": "#c62828", "review": "#e65100"}
        color = _colors.get(status, "#555")
        _icons = {"accepted": "✅ ACCEPTED", "rejected": "❌ REJECTED", "review": "⚠️ REVIEW"}
        icon = _icons.get(status, status)

        reception = real_result.get("reception")
        fmt = real_result.get("format_validation")
        content = real_result.get("content_validation")
        dup = real_result.get("duplicate_control")

        rows = [
            ("File", file_path.name),
            ("Size", f"{len(file_bytes) / 1024:.1f} KB"),
            ("MIME", reception.detected_mime if reception else "—"),
            ("SHA-256", reception.sha256[:20] + "…" if reception and reception.sha256 else "—"),
            ("Node 1", f"passed={reception.passed}" if reception else "—"),
            ("Node 2", f"passed={fmt.passed}  decision={fmt.decision}" if fmt else "not reached"),
            ("Node 3", f"passed={content.passed} ocr={content.requires_ocr}" if content else "--"),
            ("Node 4", f"passed={dup.passed}  dup={dup.is_duplicate}" if dup else "not reached"),
            ("Rejection reason", reason),
        ]

        rows_html = "".join(
            f'<tr><td style="color:#555;padding:4px 12px 4px 0;white-space:nowrap">{k}</td>'
            f'<td style="font-family:monospace;padding:4px 0">{v}</td></tr>'
            for k, v in rows
        )

        ipyd.display(
            HTML(f"""
        <div style="border:1px solid #ddd;border-radius:8px;padding:16px;
                    margin:8px 0;font-family:sans-serif;max-width:600px">
          <div style="font-weight:bold;color:{color};margin-bottom:10px">{icon}</div>
          <table style="border-collapse:collapse;width:100%">{rows_html}</table>
        </div>
        """)
        )

2026-08-06 21:54:00.531 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-real-boletin_65_2005_doc_39153 node=node1_file_reception event=passed
2026-08-06 21:54:00.535 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-real-boletin_65_2005_doc_39153 node=node2_format_validation event=passed
2026-08-06 21:54:00.564 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-real-boletin_65_2005_doc_39153 node=node3_content_validation event=passed


C:\Users\leona\source\repos\Trabajo-Integrador\src\classiflow\playground\samples


2026-08-06 21:54:03.469 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-real-boletin_65_2005_doc_39153 node=node4_duplicate_control event=passed


File,boletin_65_2005_doc_39153.pdf
Size,1597.7 KB
MIME,application/pdf
SHA-256,95c4dbddfd5ac834b275…
Node 1,passed=True
Node 2,passed=True decision=accept
Node 3,passed=True ocr=False
Node 4,passed=True dup=False
Rejection reason,—


2026-08-06 21:54:03.498 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-real-convenio_2_2013 node=node1_file_reception event=passed
2026-08-06 21:54:03.502 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-real-convenio_2_2013 node=node2_format_validation event=passed
2026-08-06 21:54:03.509 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-real-convenio_2_2013 node=node3_content_validation event=passed
2026-08-06 21:54:03.762 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-real-convenio_2_2013 node=node4_duplicate_control event=passed


...

File,DIA_A_Grupos_ACTUALIZADOS.xlsx
Size,10.0 KB
MIME,application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
SHA-256,f11ee60402bf021ce6e1…
Node 1,passed=True
Node 2,passed=True decision=accept
Node 3,passed=True ocr=False
Node 4,passed=True dup=False
Rejection reason,—


2026-08-06 21:54:03.924 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-real-ordenanza_6801_1999 node=node1_file_reception event=passed
2026-08-06 21:54:03.929 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-real-ordenanza_6801_1999 node=node2_format_validation event=passed
2026-08-06 21:54:04.098 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-real-ordenanza_6801_1999 node=node3_content_validation event=passed
2026-08-06 21:54:29.769 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-real-ordenanza_6801_1999 node=node4_duplicate_control event=passed


File,ordenanza_6801_1999.pdf
Size,14350.6 KB
MIME,application/pdf
SHA-256,6edeb07b13d30c65baf5…
Node 1,passed=True
Node 2,passed=True decision=accept
Node 3,passed=True ocr=False
Node 4,passed=True dup=False
Rejection reason,—


2026-08-06 21:54:29.785 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-real-test node=node1_file_reception event=passed
2026-08-06 21:54:29.788 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-real-test node=node2_format_validation event=passed
2026-08-06 21:54:29.792 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-real-test node=node3_content_validation event=passed
2026-08-06 21:54:31.804 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=coord-real-test node=node4_duplicate_control event=passed


File,test.txt
Size,1115.0 KB
MIME,application/octet-stream
SHA-256,7c0d04a9ce93fb812cd1…
Node 1,passed=True
Node 2,passed=True decision=accept
Node 3,passed=True ocr=False
Node 4,passed=True dup=False
Rejection reason,—


## 13 — Audit records across all nodes

Every node writes an audit entry on each run. Querying them shows the full
execution trace — which node ran, what it decided, and how long it took.

In [13]:
job_ids = [
    "coord-demo-001",  # accepted
    "coord-demo-002",  # exact duplicate
    "coord-demo-003",  # semantic duplicate
    "coord-demo-004",  # unrelated — accepted
    "coord-demo-005",  # SLM reject → review
]

async with session_factory() as session:
    repo = SqlAuditRepository(session)
    all_records = []
    for jid in job_ids:
        all_records.extend(await repo.list_for_job(jid))

print(f"{'job_id':<28} {'node':<35} {'event':<8} {'ms':>6}")
print("-" * 85)
for r in all_records:
    print(f"  {r.job_id:<26} {r.node:<35} {r.event:<8} {r.duration_ms:>5} ms")

job_id                       node                                event        ms
-------------------------------------------------------------------------------------
  coord-demo-001             node1_file_reception                passed       0 ms
  coord-demo-001             node2_format_validation             passed       0 ms
  coord-demo-001             node3_content_validation            passed      15 ms
  coord-demo-001             node4_duplicate_control             passed    4125 ms
  coord-demo-002             node1_file_reception                passed       0 ms
  coord-demo-002             node2_format_validation             passed       0 ms
  coord-demo-002             node3_content_validation            passed       0 ms
  coord-demo-002             node4_duplicate_control             failed       0 ms
  coord-demo-003             node1_file_reception                passed       0 ms
  coord-demo-003             node2_format_validation             passed       0 ms
  c

## 14 — Cleanup

In [14]:
await engine.dispose()
print("engine disposed — database file is now unlocked")

engine disposed — database file is now unlocked
